# 15 · J-space projection — {shared, dark-specific} × {in, outside J-space}

**The test.** Decompose the dark induced-shift against depression's, per layer:
`dark = shared + residual`, where `shared` = projection of `(dark−base)` onto the unit
`(depression−base)` direction and `residual` = the orthogonal dark-specific part (shown in the
local analysis to be ⊥ depression at every layer, `directions_v1/dark_minus_depression_diffvec.json`).

Each fitted [Jacobian lens](https://transformer-circuits.pub/2026/workspace/index.html) holds
per-layer `J_l` (4096×4096) transporting the layer-`l` residual into the final-layer basis —
`J_l`'s top singular subspace is *what the model actually carries to the output*; its near-null
space is representation that exists internally but never reaches the logits.

So the 2×2 is: **{shared, dark-specific} × {in J-space, outside J-space}**, at the mid band
(L16–24, where the dark-specific fraction is largest) vs late (L30–34). Run under **three lenses**
(base / dark / clinical-depression — each organism has its own fitted transport):

- shared **in**, residual **out** (base lens) → the dark-specific signature is internal dark matter
  the base transport would never surface — and if dark's *own* lens carries it, the fine-tune
  rewired the transport, not just the state. That's the clean novel result.
- both **in** → the residual is output-facing; late-layer convergence is genuinely behavioral.
- Scored two ways per (vector, lens, layer): **capture** = fraction of the vector's energy inside
  the top-k* right-singular subspace (k* = 90% spectral energy), and **gain** = `‖J·v̂‖²` relative
  to the mean over random unit vectors (chance = 1.0).

Needs: shift bundles in `DRIVE/directions_v1/` (from `06c`), lenses from HF
(`neuronpedia/jacobian-lens` + `Koalacrown/jacobian-lens-organisms`, ~3.5 GB total).
**Hardware:** any Colab GPU (T4 fine — SVDs only, no model loading); ~14 layers × 3 lenses.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U huggingface_hub
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine — all three lens repos are public)")

In [ ]:
DRIVE = mount_drive()
import pathlib, json, pickle
import numpy as np, torch
OUT = (DRIVE / "directions_v1") if DRIVE else pathlib.Path("directions_v1")
assert OUT.exists(), f"{OUT} not found — run 06c first (shift bundles)"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("directions:", OUT, "| device:", DEV)

## 2. Config
Bands: `MID` = where the dark-specific residual is proportionally largest (ratio ≈ 0.75–0.79);
`LATE` = where the raw dark↔depression cosine peaks (0.86–0.92). `ENERGY_CUT` sets k* (the
"in J-space" rank) from the singular spectrum of each `J_l`.

In [ ]:
LENSES = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
    "clinical-depression": ("Koalacrown/jacobian-lens-organisms",
                            "clinical-depression/jacobian_lens.pt"),
}
BANDS      = {"mid (16-24)": range(16, 25), "late (30-34)": range(30, 35)}
ENERGY_CUT = 0.90   # k* = smallest k holding this fraction of sum(s^2)
N_RANDOM   = 64     # random unit vectors for the gain baseline / capture band
SEED       = 0
print(f"{len(LENSES)} lenses | bands: {list(BANDS)} | k* at {ENERGY_CUT:.0%} spectral energy")

## 3. Shift bundles → per-layer {shared, residual} decomposition
`shared_L = (dark_L·û_dep_L)·û_dep_L`, `residual_L = dark_L − shared_L` (⊥ depression by
construction; the local analysis found cos(residual, dark−base) ≈ 0.4–0.8, cos(residual,
depression−base) ≈ 0 — this cell re-derives and re-checks that).

In [ ]:
def load_shift(name):
    p = OUT / f"control_vectors_shift_{name}.pkl"
    assert p.exists(), f"{p} missing — run 06c for {name}"
    return pickle.load(open(p, "rb"))["vectors"]["induced_shift"]   # {L: [4096] float32}

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(dark_s) & set(dep_s))

VECS = {}   # {L: {"shared": v, "residual": v}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u = b / np.linalg.norm(b)
    shared = float(a @ u) * u
    resid  = a - shared
    VECS[L] = {"shared": shared, "residual": resid}
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | sanity @L20:",
      f"cos(resid,dep)={VECS[20]['residual'] @ (dep_s[20]/np.linalg.norm(dep_s[20])) / np.linalg.norm(VECS[20]['residual']):+.4f}",
      f"|shared|={np.linalg.norm(VECS[20]['shared']):.1f} |resid|={np.linalg.norm(VECS[20]['residual']):.1f}")

## 4. Download lenses + per-layer SVD
One lens in memory at a time (keep only band layers). For each kept `J_l`: full SVD, then per
vector `v` the **capture curve** `c(k) = Σ_{i≤k}(v̂·V_i)²` (chance = k/4096), read at k*, and the
**gain** `‖J v̂‖²` normalized by the random-vector mean.

In [ ]:
from huggingface_hub import hf_hub_download
WANT = sorted(set().union(*[set(r) for r in BANDS.values()]) & set(SHIFT_LAYERS))
rng = np.random.default_rng(SEED)
RAND = rng.standard_normal((N_RANDOM, 4096)).astype(np.float32)
RAND /= np.linalg.norm(RAND, axis=1, keepdims=True)

RESULTS = {}   # {lens: {L: {"kstar", "s", "vectors": {name: {"capture_curve","capture_kstar","gain"}}}}}
for lname, (repo, fname) in LENSES.items():
    path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
    blob = torch.load(path, map_location="cpu", weights_only=False)
    J_all = blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians
    layers = [L for L in WANT if L in J_all]
    print(f"\n== {lname}: {len(layers)} layers {layers[0]}..{layers[-1]} ==")
    RESULTS[lname] = {}
    for L in layers:
        J = J_all[L].float().to(DEV)                      # [4096, 4096], maps layer-L resid -> final basis
        U, S, Vh = torch.linalg.svd(J, full_matrices=False)
        s2 = (S ** 2); cum = torch.cumsum(s2, 0) / s2.sum()
        kstar = int(torch.searchsorted(cum, ENERGY_CUT).item()) + 1
        Vh_np, S_np = Vh.cpu().numpy(), S.cpu().numpy()

        def score(v):
            vh = v / np.linalg.norm(v)
            comp = Vh_np @ vh                              # coords in right-singular basis
            curve = np.cumsum(comp ** 2)                   # capture at every rank
            gain = float(((S_np * comp) ** 2).sum())       # ||J v_hat||^2
            return curve, float(curve[kstar - 1]), gain

        rnd_gain = np.array([score(r)[2] for r in RAND])
        entry = {"kstar": kstar, "spectrum": S_np.astype(np.float16),
                 "random_gain_mean": float(rnd_gain.mean()), "vectors": {}}
        for vname, v in VECS[L].items():
            curve, cap, gain = score(v)
            entry["vectors"][vname] = {"capture_curve": curve.astype(np.float16),
                                       "capture_kstar": cap,
                                       "gain_rel": gain / rnd_gain.mean()}
        RESULTS[lname][L] = entry
        print(f"  L{L:2d} k*={kstar:4d} ({kstar/4096:.1%})  "
              + "  ".join(f"{n}: cap={e['capture_kstar']:.2f} gain={e['gain_rel']:.2f}x"
                          for n, e in entry["vectors"].items()))
        del J, U, S, Vh
        if DEV == "cuda": torch.cuda.empty_cache()
    del blob, J_all

## 5. The 2×2 — {shared, dark-specific} × {in, outside J-space}
Cell value = band-mean capture at k* (chance = k*/4096, printed alongside) and band-mean relative
gain (chance = 1.0). "Outside J-space" is simply 1 − capture.

In [ ]:
TABLE = {}
for lname, per_layer in RESULTS.items():
    print(f"\n===== lens: {lname} =====")
    TABLE[lname] = {}
    for bname, rng_ in BANDS.items():
        Ls = [L for L in rng_ if L in per_layer]
        if not Ls: continue
        chance = np.mean([per_layer[L]["kstar"] for L in Ls]) / 4096
        row = {}
        for vname in ("shared", "residual"):
            cap  = np.mean([per_layer[L]["vectors"][vname]["capture_kstar"] for L in Ls])
            gain = np.mean([per_layer[L]["vectors"][vname]["gain_rel"] for L in Ls])
            row[vname] = {"capture": float(cap), "outside": float(1 - cap), "gain_rel": float(gain)}
        TABLE[lname][bname] = {"chance_capture": float(chance), **row}
        print(f"  {bname:14s} chance={chance:.2f} | "
              f"shared: in={row['shared']['capture']:.2f} out={row['shared']['outside']:.2f} gain={row['shared']['gain_rel']:.2f}x | "
              f"dark-specific: in={row['residual']['capture']:.2f} out={row['residual']['outside']:.2f} gain={row['residual']['gain_rel']:.2f}x")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, len(RESULTS), figsize=(6 * len(RESULTS), 9), squeeze=False)
for j, (lname, per_layer) in enumerate(RESULTS.items()):
    ax = axes[0][j]                                   # per-layer capture at k*
    Ls = sorted(per_layer)
    for vname, col in (("shared", "#6b7280"), ("residual", "#7c3aed")):
        ax.plot(Ls, [per_layer[L]["vectors"][vname]["capture_kstar"] for L in Ls],
                "-o", color=col, label={"shared": "shared", "residual": "dark-specific"}[vname])
    ax.plot(Ls, [per_layer[L]["kstar"] / 4096 for L in Ls], "--", color="k", lw=.8, label="chance (k*/4096)")
    ax.set_title(f"{lname}: in-J-space fraction @ k*"); ax.set_xlabel("layer"); ax.set_ylim(0, 1)
    ax.grid(alpha=.3); ax.legend(fontsize=8)

    ax = axes[1][j]                                   # per-layer relative transport gain
    for vname, col in (("shared", "#6b7280"), ("residual", "#7c3aed")):
        ax.plot(Ls, [per_layer[L]["vectors"][vname]["gain_rel"] for L in Ls], "-o", color=col)
    ax.axhline(1, color="k", lw=.8, ls="--")
    ax.set_title(f"{lname}: transport gain ‖J v̂‖² vs random"); ax.set_xlabel("layer")
    ax.set_yscale("log"); ax.grid(alpha=.3, which="both")
fig.suptitle("J-space test: {shared, dark-specific} × {in, outside}", y=1.0, fontsize=14)
fig.tight_layout()
fig.savefig(OUT / "jspace_projection.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Persist (`DRIVE/directions_v1/`)

In [ ]:
meta = {"energy_cut": ENERGY_CUT, "n_random": N_RANDOM, "seed": SEED,
        "bands": {k: list(v) for k, v in BANDS.items()}, "lenses": {k: v[0] + "/" + v[1] for k, v in LENSES.items()},
        "decomposition": "dark_shift = shared(along unit dep_shift) + residual(orthogonal); per layer",
        "table_2x2": TABLE,
        "per_layer": {ln: {int(L): {"kstar": e["kstar"], "random_gain_mean": e["random_gain_mean"],
                                     "vectors": {vn: {"capture_kstar": ve["capture_kstar"],
                                                       "gain_rel": ve["gain_rel"]}
                                                 for vn, ve in e["vectors"].items()}}
                            for L, e in pl.items()} for ln, pl in RESULTS.items()}}
json.dump(meta, open(OUT / "jspace_projection.json", "w"), indent=2)
np.savez(OUT / "jspace_projection_curves.npz",
         **{f"{ln}__L{L}__{vn}__curve": e["vectors"][vn]["capture_curve"]
            for ln, pl in RESULTS.items() for L, e in pl.items() for vn in e["vectors"]},
         **{f"{ln}__L{L}__spectrum": e["spectrum"] for ln, pl in RESULTS.items() for L, e in pl.items()})
print("saved:", OUT / "jspace_projection.json")
print("saved:", OUT / "jspace_projection_curves.npz")
print("saved:", OUT / "jspace_projection.png")

## Reading the result
- **base lens, dark-specific residual:** capture ≪ shared (esp. mid band) → the dark signature is
  representational dark matter under the base transport. Then check the **dark lens** on the same
  vector: if capture/gain jump, the fine-tune rewired `J` to carry it — the headline claim.
- **Both high everywhere:** the residual is output-facing; "orthogonal to depression" is a
  behavioral axis, not a hidden one. Still novel, different framing.
- **Gain vs capture disagree:** gain weights by `s²`, so gain≫1 with modest capture = the vector
  rides a few very-high-`s` directions; report both numbers.
- Follow-ups: same 2×2 under the **depression lens** (already computed above) tells you whether
  depression's transport is blind to the dark residual — the symmetric control; and transporting
  `J_l · residual` to vocab space (11-lab machinery) names *what* the dark-specific direction says
  when it does reach the output.